# Objective
Assess whether the company's rapid client growth (2021–2025) is profitable and sustainable, by analysing revenue/cost trends, client segments, and satisfaction drivers to recommend where to focus retention and pricing efforts.

In [87]:
import pandas as pd
import plotly.express as px
import plotly.graph_objs as go
import numpy as np

In [88]:
profiles = pd.read_excel("..\AY2627_S1_CA2 datasets & meta data\CA2 datasets & meta data\Client profiles.xlsx")
profiles.head()

,CLIENT ID,TYPE,SG-BASED,OVERSEAS,COUNTRY OF OVERSEAS CLIENT,COMMENCEMENT DATE,STAFF STRENGTH,SECTOR
0,G1-001,Govt,Yes,NaN,,2020-01-19 00:00:00,More than 200,Finance
1,G1-002,Govt,Yes,NaN,,2020-05-19 00:00:00,More than 200,Security
2,G1-003,Govt,Yes,NaN,,2021-07-18 00:00:00,> 200,Transportation
3,G1-005,Govt,Y,NaN,NaN,2022-02-11 00:00:00,> 200,Security
4,G1-006,Govt,Y,NaN,NaN,2022-03-18 00:00:00,50 ~ 200,Legal


Things to clean:\
    - columns using different values to represent the same info(Sg-based,staff strength)\
    - getting rid of missing values (Country of overseas client)

In [89]:
profiles[['COUNTRY OF OVERSEAS CLIENT','OVERSEAS','SG-BASED']].value_counts()

COUNTRY OF OVERSEAS CLIENT  OVERSEAS  SG-BASED
Malaysia                    Yes                   6
                            Y                     3
India                       Yes                   1
China                       Y                     1
Indonesia                   Yes                   1
Malaysian                   Y                     1
Name: count, dtype: int64

In [90]:
# ---- Clean Client Profiles ----
# Strip stray leading/trailing whitespace on text columns
obj_cols = ['SG-BASED', 'OVERSEAS', 'COUNTRY OF OVERSEAS CLIENT',
            'STAFF STRENGTH', 'SECTOR', 'TYPE', 'CLIENT ID']
for c in obj_cols:
    profiles[c] = profiles[c].apply(lambda v: v.strip() if isinstance(v, str) else v)

# SG-BASED / OVERSEAS: standardise to explicit Y / N ('Yes' and 'Y' -> 'Y', blanks -> 'N')
def yes_no(v):
    return 'Y' if str(v).strip().upper() in ('Y', 'YES') else 'N'
profiles['SG-BASED'] = profiles['SG-BASED'].apply(yes_no)
profiles['OVERSEAS'] = profiles['OVERSEAS'].apply(yes_no)

# COUNTRY OF OVERSEAS CLIENT: unify blanks to NaN, fix 'Malaysian' -> 'Malaysia'
profiles['COUNTRY OF OVERSEAS CLIENT'] = (
    profiles['COUNTRY OF OVERSEAS CLIENT']
    .replace(r'^\s*$', np.nan, regex=True)       # empty / whitespace-only -> NaN
    .replace('Malaysian', 'Malaysia')
)

# Derived country: domestic clients -> 'Singapore', otherwise the overseas country
profiles['country'] = profiles['COUNTRY OF OVERSEAS CLIENT'].fillna('Singapore')

# STAFF STRENGTH: normalise synonym labels, then derive an org_size bucket.
# Metadata rule: 1~49 = Small, 50~200 = Medium, >200 = Large.
# 'Less than 200' is ambiguous (spans Small & Medium) -> kept as 'Unspecified'.
profiles['STAFF STRENGTH'] = profiles['STAFF STRENGTH'].replace(
    {'More than 200': '> 200', 'Less than 49': '1 ~ 49'})
size_map = {'1 ~ 49': 'Small', '50 ~ 200': 'Medium',
            '> 200': 'Large', 'Less than 200': 'Unspecified'}
profiles['org_size'] = profiles['STAFF STRENGTH'].map(size_map)
assert profiles['org_size'].notna().all(), \
    profiles.loc[profiles['org_size'].isna(), 'STAFF STRENGTH'].unique()

# COMMENCEMENT DATE: mix of datetime & string -> uniform datetime64
profiles['COMMENCEMENT DATE'] = pd.to_datetime(profiles['COMMENCEMENT DATE'], errors='coerce')
assert profiles['COMMENCEMENT DATE'].notna().all(), "Unparseable COMMENCEMENT DATE remains"

profiles.head()

,CLIENT ID,TYPE,SG-BASED,OVERSEAS,COUNTRY OF OVERSEAS CLIENT,COMMENCEMENT DATE,STAFF STRENGTH,SECTOR,country,org_size
0,G1-001,Govt,Y,N,NaN,2020-01-19,> 200,Finance,Singapore,Large
1,G1-002,Govt,Y,N,NaN,2020-05-19,> 200,Security,Singapore,Large
2,G1-003,Govt,Y,N,NaN,2021-07-18,> 200,Transportation,Singapore,Large
3,G1-005,Govt,Y,N,NaN,2022-02-11,> 200,Security,Singapore,Large
4,G1-006,Govt,Y,N,NaN,2022-03-18,50 ~ 200,Legal,Singapore,Medium


In [91]:
profiles['STAFF STRENGTH'].value_counts()

STAFF STRENGTH
> 200            62
50 ~ 200         57
1 ~ 49           22
Less than 200     7
Name: count, dtype: int64

In [92]:
profiles['SG-BASED'].value_counts()

SG-BASED
Y    114
N     34
Name: count, dtype: int64

In [93]:
profiles.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148 entries, 0 to 147
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   CLIENT ID                   148 non-null    object        
 1   TYPE                        148 non-null    object        
 2   SG-BASED                    148 non-null    object        
 3   OVERSEAS                    148 non-null    object        
 4   COUNTRY OF OVERSEAS CLIENT  34 non-null     object        
 5   COMMENCEMENT DATE           148 non-null    datetime64[ns]
 6   STAFF STRENGTH              148 non-null    object        
 7   SECTOR                      148 non-null    object        
 8   country                     148 non-null    object        
 9   org_size                    148 non-null    object        
dtypes: datetime64[ns](1), object(9)
memory usage: 11.7+ KB


In [94]:
survey = pd.read_excel("..\AY2627_S1_CA2 datasets & meta data\CA2 datasets & meta data\Client survey.xlsx")
survey.head()

,YEAR,CLIENT ID,PRESALES AND PARTNERSHIP,TECHNICAL EXPERTISE,PROJECT DELIVERY,POST-SALES SUPPORT,NPS RATING
0,2021,G1-001,3,4,4,5,8
1,2021,G1-002,3,4,4,2,7
2,2021,G1-003,5,4,3,5,8
3,2021,G1-001,3,4,4,5,8
4,2021,G1-002,3,4,4,2,7


In [95]:
survey['CLIENT ID'].value_counts()

CLIENT ID
G1-001    9
G1-002    9
G1-003    9
G1-006    7
N1-001    5
         ..
P3-027    1
P3-028    1
P3-029    1
P3-030    1
P3-031    1
Name: count, Length: 143, dtype: int64

In [96]:
survey.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 432 entries, 0 to 431
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   YEAR                      432 non-null    int64 
 1   CLIENT ID                 432 non-null    object
 2   PRESALES AND PARTNERSHIP  432 non-null    int64 
 3   TECHNICAL EXPERTISE       432 non-null    int64 
 4   PROJECT DELIVERY          432 non-null    int64 
 5   POST-SALES SUPPORT        432 non-null    int64 
 6   NPS RATING                432 non-null    int64 
dtypes: int64(6), object(1)
memory usage: 23.8+ KB


In [97]:
survey.isnull().sum()

YEAR                        0
CLIENT ID                   0
PRESALES AND PARTNERSHIP    0
TECHNICAL EXPERTISE         0
PROJECT DELIVERY            0
POST-SALES SUPPORT          0
NPS RATING                  0
dtype: int64

In [98]:
# ---- Clean Client Survey ----
# Drop 30 exact duplicate rows, then drop orphan CLIENT IDs not in Client Profiles
survey = survey.drop_duplicates()
survey = survey[survey['CLIENT ID'].isin(profiles['CLIENT ID'])].reset_index(drop=True)

# Derived NPS category per respondent (Meta Data thresholds)
def nps_category(score):
    if score >= 9:
        return 'Promoter'    # 9-10
    elif score >= 7:
        return 'Passive'     # 7-8
    return 'Detractor'       # 6 or less
survey['nps_category'] = survey['NPS RATING'].apply(nps_category)

survey.head()

,YEAR,CLIENT ID,PRESALES AND PARTNERSHIP,TECHNICAL EXPERTISE,PROJECT DELIVERY,POST-SALES SUPPORT,NPS RATING,nps_category
0,2021,G1-001,3,4,4,5,8,Passive
1,2021,G1-002,3,4,4,2,7,Passive
2,2021,G1-003,5,4,3,5,8,Passive
3,2021,N1-001,4,4,4,3,8,Passive
4,2021,N1-002,5,3,4,3,7,Passive


In [99]:
RC = pd.read_excel("..\AY2627_S1_CA2 datasets & meta data\CA2 datasets & meta data\Revenue and COGS.xlsx")
RC.head()

,CLIENT ID,YEAR,REVENUE,HARDWARE,SOFTWARE,MANPOWER
0,G1-001,2021,148180,17321,25562,41201
1,G1-001,2022,197313,23604,32875,50085
2,G1-001,2023,349899,51332,71695,89089
3,G1-001,2024,268821,40969,56452,76545
4,G1-001,2025,138115,14516,20889,28968


In [100]:
# ---- Clean Revenue and COGS ----
# Drop 21 exact duplicate rows, then drop orphan CLIENT IDs not in Client Profiles
# (P1-010, P1-018, P1-035 -> metadata says such records can be ignored)
RC = RC.drop_duplicates()
RC = RC[RC['CLIENT ID'].isin(profiles['CLIENT ID'])].reset_index(drop=True)

# Derived cost / profitability metrics (formulas from Meta Data)
RC['cogs'] = RC['HARDWARE'] + RC['SOFTWARE'] + RC['MANPOWER']
RC['gross_profit'] = RC['REVENUE'] - RC['cogs']
RC['gross_margin_pct'] = (RC['gross_profit'] / RC['REVENUE'] * 100).round(2)
RC['gross_markup_pct'] = (RC['gross_profit'] / RC['cogs'] * 100).round(2)
RC['loss'] = RC['gross_profit'] < 0   # negative gross profit (all NPO clients) -> flagged, kept

RC.head()

,CLIENT ID,YEAR,REVENUE,HARDWARE,SOFTWARE,MANPOWER,cogs,gross_profit,gross_margin_pct,gross_markup_pct,loss
0,G1-001,2021,148180,17321,25562,41201,84084,64096,43.26,76.23,False
1,G1-001,2022,197313,23604,32875,50085,106564,90749,45.99,85.16,False
2,G1-001,2023,349899,51332,71695,89089,212116,137783,39.38,64.96,False
3,G1-001,2024,268821,40969,56452,76545,173966,94855,35.29,54.53,False
4,G1-001,2025,138115,14516,20889,28968,64373,73742,53.39,114.55,False


In [101]:
RC.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 392 entries, 0 to 391
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   CLIENT ID         392 non-null    object 
 1   YEAR              392 non-null    int64  
 2   REVENUE           392 non-null    int64  
 3   HARDWARE          392 non-null    int64  
 4   SOFTWARE          392 non-null    int64  
 5   MANPOWER          392 non-null    int64  
 6   cogs              392 non-null    int64  
 7   gross_profit      392 non-null    int64  
 8   gross_margin_pct  392 non-null    float64
 9   gross_markup_pct  392 non-null    float64
 10  loss              392 non-null    bool   
dtypes: bool(1), float64(2), int64(7), object(1)
memory usage: 31.1+ KB


In [102]:
RC.isnull().sum()

CLIENT ID           0
YEAR                0
REVENUE             0
HARDWARE            0
SOFTWARE            0
MANPOWER            0
cogs                0
gross_profit        0
gross_margin_pct    0
gross_markup_pct    0
loss                0
dtype: int64

In [103]:
profiles.head()

,CLIENT ID,TYPE,SG-BASED,OVERSEAS,COUNTRY OF OVERSEAS CLIENT,COMMENCEMENT DATE,STAFF STRENGTH,SECTOR,country,org_size
0,G1-001,Govt,Y,N,NaN,2020-01-19,> 200,Finance,Singapore,Large
1,G1-002,Govt,Y,N,NaN,2020-05-19,> 200,Security,Singapore,Large
2,G1-003,Govt,Y,N,NaN,2021-07-18,> 200,Transportation,Singapore,Large
3,G1-005,Govt,Y,N,NaN,2022-02-11,> 200,Security,Singapore,Large
4,G1-006,Govt,Y,N,NaN,2022-03-18,50 ~ 200,Legal,Singapore,Medium


In [104]:
survey.head()

,YEAR,CLIENT ID,PRESALES AND PARTNERSHIP,TECHNICAL EXPERTISE,PROJECT DELIVERY,POST-SALES SUPPORT,NPS RATING,nps_category
0,2021,G1-001,3,4,4,5,8,Passive
1,2021,G1-002,3,4,4,2,7,Passive
2,2021,G1-003,5,4,3,5,8,Passive
3,2021,N1-001,4,4,4,3,8,Passive
4,2021,N1-002,5,3,4,3,7,Passive


In [105]:
RC.head()

,CLIENT ID,YEAR,REVENUE,HARDWARE,SOFTWARE,MANPOWER,cogs,gross_profit,gross_margin_pct,gross_markup_pct,loss
0,G1-001,2021,148180,17321,25562,41201,84084,64096,43.26,76.23,False
1,G1-001,2022,197313,23604,32875,50085,106564,90749,45.99,85.16,False
2,G1-001,2023,349899,51332,71695,89089,212116,137783,39.38,64.96,False
3,G1-001,2024,268821,40969,56452,76545,173966,94855,35.29,54.53,False
4,G1-001,2025,138115,14516,20889,28968,64373,73742,53.39,114.55,False


In [106]:
# ---- Export cleaned datasets ----
import os
os.makedirs('cleaned', exist_ok=True)
profiles.to_csv('cleaned/client_profiles_clean.csv', index=False)
RC.to_csv('cleaned/revenue_cogs_clean.csv', index=False)
survey.to_csv('cleaned/client_survey_clean.csv', index=False)
print('Exported cleaned files:', os.listdir('cleaned'))

Exported cleaned files: ['client_profiles_clean.csv', 'client_survey_clean.csv', 'revenue_cogs_clean.csv']


## DB 1 (Elliot)

#### Graph 1

In [107]:
client_year_df = (
    RC
    .merge(profiles, on="CLIENT ID", how="left")
    .merge(survey, on=["CLIENT ID", "YEAR"], how="inner")
)

In [108]:
nps_categories = ["Detractor", "Passive", "Promoter"]              # legend order
category_colours = {"Detractor": "#d62728", "Passive": "#9aa0a6", "Promoter": "#2ca02c"}

# One least-squares line per YEAR, in plain pandas:
#   slope = cov(x, y) / var(x),  intercept = mean(y) - slope * mean(x)
# Two endpoints are enough to draw a straight line.
def yearly_trend(year_group):
    slope = year_group["NPS RATING"].cov(year_group["gross_margin_pct"]) / year_group["NPS RATING"].var()
    intercept = year_group["gross_margin_pct"].mean() - slope * year_group["NPS RATING"].mean()
    x = pd.Series([4, 10])                                        # span the full NPS scale
    return pd.DataFrame({"NPS RATING": x, "gross_margin_pct": intercept + slope * x})

trend = (client_year_df.groupby("YEAR")[["NPS RATING", "gross_margin_pct"]]
           .apply(yearly_trend)
           .reset_index())
trend["SERIES"] = "Trend for this year"
trend

,YEAR,level_1,NPS RATING,gross_margin_pct,SERIES
0,2021,0,4,43.593982,Trend for this year
1,2021,1,10,42.722741,Trend for this year
2,2022,0,4,48.722021,Trend for this year
3,2022,1,10,42.916064,Trend for this year
4,2023,0,4,44.710546,Trend for this year
5,2023,1,10,38.738449,Trend for this year
6,2024,0,4,46.965845,Trend for this year
7,2024,1,10,39.626624,Trend for this year
8,2025,0,4,50.496931,Trend for this year
9,2025,1,10,39.446947,Trend for this year


In [109]:
fig = px.scatter(
    client_year_df,
    x="NPS RATING",
    y="gross_margin_pct",
    size="REVENUE",                       # bubble size = deal size
    color="nps_category",
    color_discrete_map=category_colours,  # red = detractor, grey = passive, green = promoter
    category_orders={"nps_category": nps_categories},
    animation_frame="YEAR",               # -> year slider + play button
    animation_group="CLIENT ID",          # same client morphs smoothly between years
    hover_name="CLIENT ID",
    hover_data={"SECTOR": True, "TYPE": True, "org_size": True,
                "REVENUE": ":,.0f", "gross_margin_pct": ":.1f"},
    size_max=45,
    range_x=[3.5, 10.5],                 # fixed axes so the animation doesn't jump
    range_y=[-40, 70],
    title="Do happier client earn bring in more profit?",
)

# Animated trendline: px.line gives us one line trace per year, which we splice into
# the scatter's frames so the fit is redrawn every time the year slider moves.
trend_fig = px.line(trend, x="NPS RATING", y="gross_margin_pct",
                    animation_frame="YEAR", color="SERIES",
                    color_discrete_sequence=["#333"])

fig.add_traces(trend_fig.data)                     # line for the year the slider starts on
lines_by_year = {frame.name: frame.data for frame in trend_fig.frames}
for frame in fig.frames:                           # ...and for every other year
    frame.data = frame.data + lines_by_year[frame.name]

# Reference lines
overall_margin = client_year_df["gross_profit"].sum() / client_year_df["REVENUE"].sum() * 100
fig.add_hline(y=overall_margin, line_dash="dot", line_color="#555",
              annotation_text=f"Overall margin {overall_margin:.1f}%",
              annotation_position="top left")
fig.add_hline(y=0, line_color="#d62728", line_width=1,
              annotation_text="Break-even", annotation_position="bottom right")

fig.update_layout(
    xaxis_title="NPS Rating (0-10)",
    yaxis_title="Gross Margin (%)",
    height = 700
)

fig.show()

### Insights

1. Happier clients = thinner margins. Gross margin falls from 46.6% (Detractors) -> 43.4% (Passives) -> 40.6% (Promoters), even though Promoters bring the highest revenue (~$390k vs ~$279k).
2. The gap is widening every year. Trend line was flat in 2021 (−0.15% margin per NPS point) but steepens to −1.85% by 2025. Top-rated clients now earn ~10% less margin than low-rated ones.
3. All 5 loss-making engagements are high-NPS charity clients. They rate us 9+ while we lose money having them as cilents (e.g. N1-002, N1-007 for two years running).

#### Graph 2

In [110]:
cogs_df = RC.merge(profiles, on="CLIENT ID", how="left")

cogs_components = ["HARDWARE", "SOFTWARE", "MANPOWER"]                       # dataset field names
component_colours = {"HARDWARE": "#8c6d31", "SOFTWARE": "#6b6ecf", "MANPOWER": "#e6550d"}
client_type_options = ["All clients", "Private", "Govt", "NPO"]

In [111]:
def yearly_costs(data):
    """Yearly totals per COGS component, weighted margin, and cost shares."""
    yearly = data.groupby("YEAR")[cogs_components + ["REVENUE", "gross_profit"]].sum().reset_index()
    yearly["margin"] = yearly["gross_profit"] / yearly["REVENUE"] * 100
    cogs_total = yearly[cogs_components].sum(axis=1)
    for component in cogs_components:
        yearly[component + " share"] = yearly[component] / cogs_total * 100   # used in hover text
    return yearly

yearly_by_type = {
    client_type: yearly_costs(cogs_df if client_type == "All clients" else cogs_df[cogs_df["TYPE"] == client_type])
    for client_type in client_type_options
}

In [112]:
fig = go.Figure()

traces_per_type = len(cogs_components) + 1                  # 3 cost bars + 1 margin line

for client_type in client_type_options:
    yearly = yearly_by_type[client_type]
    show = client_type == client_type_options[0]
    for component in cogs_components:
        fig.add_trace(go.Bar(
            x=yearly["YEAR"], y=yearly[component],
            name=component.title(),
            marker_color=component_colours[component],
            visible=show,
            legendgroup=component,                       # one legend entry per component
            customdata=yearly[component + " share"],
            hovertemplate=component.title() + ": $%{y:,.0f} (%{customdata:.0f}% of COGS)",
        ))
    fig.add_trace(go.Scatter(
        x=yearly["YEAR"], y=yearly["margin"],
        name="Gross Margin %",
        mode="lines+markers+text",
        text=yearly["margin"].round(1).astype(str) + "%",
        textposition="top center",
        line=dict(color="#222", width=3),
        marker=dict(size=9),
        visible=show,
        yaxis="y2",                              # secondary axis: % not $
        hovertemplate="Margin: %{y:.1f}%",
    ))

# Dropdown to switch which client type's traces are visible (title updates to match).
buttons = [dict(label=selected_type, method="update",
                args=[{"visible": [trace_type == selected_type
                                    for trace_type in client_type_options
                                    for _ in range(traces_per_type)]},
                      {"title.text": f"COGS composition and gross margin for {selected_type}"}])
           for selected_type in client_type_options]

fig.update_layout(
    barmode="stack",
    title=f"COGS composition and gross margin for {client_type_options[0]}",
    xaxis=dict(title="Year", tickmode="linear", dtick=1),
    yaxis=dict(title="Cost of Goods Sold ($)", tickformat="~s"),   # 10M, 20M ...
    yaxis2=dict(title="Gross Margin (%)", overlaying="y", side="right",
                range=[0, 60], showgrid=False),
    updatemenus=[dict(buttons=buttons, x=1.0, xanchor="right",
                      y=1.15, yanchor="top")],
    hovermode="x",
    legend=dict(orientation="h", y=-0.18),
    height = 700
)

fig.show()

### Insights
1. Manpower is the biggest cost every year, and it keeps growing. From 41% of COGS in 2021 to 44% in 2025, it's also growing faster than revenue (8.2x vs 7.8x), which means the company is getting more labour-heavy as it scales.
2. Despite revenue growing almost 8x, overall margin has stayed steady at around 44%. Growth hasn't hurt profitability.
3. Switching the filter to NPO shows a completely different picture: margins of only 10 to 25%, compared to ~44% for Private and Govt clients.

### Graph 3

In [113]:
sector_df = (
    RC
    .merge(profiles, on="CLIENT ID", how="left")
    .merge(survey, on=["CLIENT ID", "YEAR"], how="left")
)

In [114]:
sector_order = (sector_df.groupby("SECTOR")["gross_margin_pct"].median()
                  .sort_values().index.tolist())
n_by_sector = sector_df["SECTOR"].value_counts()
sector_labels = {s: f"{s}<br>(n={n_by_sector[s]})" for s in sector_order}
sector_df["sector_label"] = sector_df["SECTOR"].map(sector_labels)
sector_label_order = [sector_labels[s] for s in sector_order]

sector_df["point_text"] = sector_df["CLIENT ID"] + " · " + sector_df["YEAR"].astype(str)

In [115]:
overall_margin = sector_df["gross_profit"].sum() / sector_df["REVENUE"].sum() * 100

metrics = {
    "Gross Margin %": dict(col="gross_margin_pct", axis_type="linear",
                           fmt=".1f", suffix="%", refline=overall_margin),
    "Revenue":        dict(col="REVENUE", axis_type="log",   # log: revenue is heavily skewed
                           fmt=",.0f", suffix=" $", refline=None),
    "NPS Rating":     dict(col="NPS RATING", axis_type="linear",
                           fmt=".0f", suffix="", refline=None),
}
default_metric = next(iter(metrics))                 # first entry = initial view

In [116]:
charity_mask = sector_df["SECTOR"] == "Charity"
sector_groups = [
    ("Other sectors", "#7f8fa6", ~charity_mask),
    ("Charity", "#d62728", charity_mask),
]

fig = go.Figure()

for name, metric in metrics.items():
    show = name == default_metric              # initial view
    for group_name, group_colour, group_mask in sector_groups:
        subset = sector_df[group_mask]
        fig.add_trace(go.Box(
            x=subset["sector_label"], y=subset[metric["col"]],
            name=group_name,
            marker_color=group_colour,
            boxpoints="all", jitter=0.5, pointpos=0,   # points ON the box, jittered
            marker=dict(size=4, opacity=0.5),
            line=dict(width=1.5),
            visible=show,
            text=subset["point_text"],
            hovertemplate="%{text}<br>%{y:" + metric["fmt"] + "}" + metric["suffix"],
        ))

In [117]:
def ref_shapes(metric):
    if metric["refline"] is None:
        return []
    return [dict(type="line", xref="paper", x0=0, x1=1,
                 y0=metric["refline"], y1=metric["refline"],
                 line=dict(dash="dot", color="#555", width=2))]

buttons = [dict(label=name, method="update",
                args=[{"visible": [n == name for n in metrics for _ in range(len(sector_groups))]},
                      {"yaxis.title.text": name,
                       "yaxis.type": metric["axis_type"],
                       "shapes": ref_shapes(metric),
                       "title.text": f"Distribution of {name} by sector"}])
           for name, metric in metrics.items()]

In [118]:
fig.update_layout(
    title=f"Distribution of {default_metric} by sector",
    xaxis=dict(title="Sector (ordered by median margin, worst to best)",
               categoryorder="array", categoryarray=sector_label_order),
    yaxis=dict(title=default_metric),
    shapes=ref_shapes(metrics[default_metric]),
    updatemenus=[dict(type="buttons", direction="right",   # renders as a radio row
                      buttons=buttons, x=0.5, xanchor="center",
                      y=1.12, yanchor="top", showactive=True)],
    boxmode="group",
    legend=dict(orientation="h", y=-0.25),
    height = 700
)

fig.show()

### Insights
1. In terms of profits, the charity sector's entire margin distribution sits below every other sector (median 18.6% vs 41.8–47.3% for the rest).
2. In revenue, the charity sector is financially tiny (around $1.0M of $135M total, less than 1% of total revenue). So the losses are small in dollar terms, and the company can decide calmly whether to keep supporting these clients cheaply or start charging them properly.
3. In terms of NPS, charity flips to one of the highest-rated sectors (median 9). Confirming that the clients who like us most are the ones we earn the least from.

## DB 2 (Zafran)

#### Graph 1

Dimension vs NPS rating boxplot

In [119]:
# Columns we want to be able to switch between
metrics = [
    'NPS RATING',
    'PRESALES AND PARTNERSHIP',
    'TECHNICAL EXPERTISE',
    'PROJECT DELIVERY',
    'POST-SALES SUPPORT'
]

# Compute yearly average for each metric
yearly_avg = survey.groupby('YEAR')[metrics].mean().reset_index()

# Assign a distinct color to each metric
colors = {
    'NPS RATING': '#636EFA',
    'PRESALES AND PARTNERSHIP': '#EF553B',
    'TECHNICAL EXPERTISE': '#00CC96',
    'PROJECT DELIVERY': '#AB63FA',
    'POST-SALES SUPPORT': '#FFA15A'
}

# Start with the first metric as the default line
fig = px.line(
    yearly_avg,
    x='YEAR',
    y=metrics[0],
    markers=True,
    title=f'Average {metrics[0]} by Year'
)
fig.update_traces(line_color=colors[metrics[0]], marker=dict(color=colors[metrics[0]]))

# Build one dropdown button per metric.
# Each button swaps the y-data, line/marker color, and updates the axis title/chart title.
buttons = []
for metric in metrics:
    buttons.append(
        dict(
            label=metric.title(),
            method='update',
            args=[
                {'y': [yearly_avg[metric]],                        # new y data for the single trace
                 'line.color': [colors[metric]],
                 'marker.color': [colors[metric]]},
                {'title': f'Average {metric.title()} by Year',
                 'yaxis': {'title': metric.title()}}
            ]
        )
    )

fig.update_layout(
    updatemenus=[
        dict(
            buttons=buttons,
            direction='down',
            showactive=True,
            x=1.25, xanchor='right',
            y=1.25, yanchor='top'
        )
    ],
    yaxis_title=metrics[0].title(),
    xaxis_title='Year'
)

fig.show()

### Insights
1. Average NPS Rating continues to rise fairly consistently every year
2. Most other metrics follow the same general trajectory as NPS Rating with Project Delivery following almost to a tee
3. Presales And Partnership and Post-sales support seems to be our weakness so we may need to bring attention to that in future years

#### Graph 2

NPS CATEGORY BAR CHART 

The ratio of nps category in a sector + ratio of npc category in all sectors in a given year

In [120]:
merged_df1 = pd.merge(profiles, survey, on='CLIENT ID', how='inner')
NPSPerYEAR = merged_df1.groupby(['YEAR'])['nps_category'].value_counts().reset_index()
NPSPerYEAR

,YEAR,nps_category,count
0,2021,Passive,19
1,2021,Detractor,5
2,2021,Promoter,5
3,2022,Passive,22
4,2022,Promoter,14
5,2022,Detractor,6
6,2023,Passive,44
7,2023,Promoter,24
8,2023,Detractor,6
9,2024,Promoter,52


In [121]:
trace_year_map = []  # tracks year for each per-year trace; None for "combined" traces

years = NPSPerYEAR['YEAR'].unique()
fig = go.Figure()

# Add pie chart of all years 
for y in years:
    sub = NPSPerYEAR[NPSPerYEAR['YEAR'] == y]
    fig.add_trace(go.Pie(labels=sub['nps_category'], values=sub['count'], name=str(y), hole=.2, visible=(y == years[0]), marker=dict(colors=['#636EFA','00CC96','#EF553B'])))
    trace_year_map.append(y)

# slider specifics 
steps = []
for yr in years:
    visibility = [ty == yr for ty in trace_year_map]
    steps.append(dict(
        method="update",
        args=[{"visible": visibility}, {"title": f"NPS Category Ratio breakdown — {yr}"}],
        label=str(yr),
    ))

# add the slider as interactive feature, change the dimensions of the fig to make 
fig.update_layout(
    sliders=[dict(active=0, currentvalue={"prefix": "Year: "}, steps=steps)],
    autosize=True,
    height=500,
    width=1100,
)

fig.update_layout(title='NPS Category Ratio breakdown - 2021')
fig.show()

## insights
1. From this chart, it shows that client satisfaction is going up through the years and by 2025, promoters make up more than 50%
2. Also, overall the percentage of detractors have gone down with them making up only ~7.5% of total clients in 2025
3. This may be due to the fact that the longer we work with our clients, the more opportunities for us to prove our worth to them

#### Graph 3

NPS Rating distribution by SECTOR violin

In [122]:
merged_df1 = pd.merge(profiles, survey, on='CLIENT ID', how='inner')
merged_df1 = merged_df1[['SECTOR','NPS RATING','YEAR']]

In [123]:
import plotly.graph_objects as go

fig = go.Figure()

years = merged_df1['YEAR'].unique()
categories = merged_df1['SECTOR'].unique()

trace_year_map = []  # tracks year for each per-year trace; None for "combined" traces

# per-year traces
for yr in years:
    for cat in categories:
        subset = merged_df1[(merged_df1['YEAR'] == yr) & (merged_df1['SECTOR'] == cat)]
        fig.add_trace(
            go.Box(
                x=subset['SECTOR'],
                y=subset['NPS RATING'],
                name=cat,
                visible=(yr == years[0]),
            )
        )
        trace_year_map.append(yr)

# combined "All years" traces — one per sector, all years pooled together
for cat in categories:
    subset = merged_df1[merged_df1['SECTOR'] == cat]
    fig.add_trace(
        go.Box(
            x=subset['SECTOR'],
            y=subset['NPS RATING'],
            name=cat,
            visible=False,  # hidden until "All" is selected
        )
    )
    trace_year_map.append(None)  # marks this as a combined trace

# slider steps — one per year
steps = []
for yr in years:
    visibility = [ty == yr for ty in trace_year_map]
    steps.append(dict(
        method="update",
        args=[{"visible": visibility}, {"title": f"NPS Rating by Sector — {yr}"}],
        label=str(yr),
    ))

# "All" step — shows only the combined traces
steps.append(dict(
    method="update",
    args=[{"visible": [ty is None for ty in trace_year_map]}, {"title": "NPS Rating by Sector — All Years"}],
    label="All",
))

fig.update_layout(
    sliders=[dict(active=0, currentvalue={"prefix": "Year: "}, steps=steps)],
    autosize=True,
    title='Dimension against NPS RATING Boxplot',
    height=500,
    width=1100,
)

fig.show()

## Insights
1. Looking at all years combined, I can see that we are lacking for many of the sectors suck as Healthcare, Manufacturing and Logistic
2. For Finance, Transportation, Legal and Charity, we do reasonably well satisfying their needs with all of them having a minimum rating above 5
3. Overall, I'd say that we should distribute our attention across all sectors to avoid dissatisfaction at this level that still continues to 2025 with some sectors satisfaction being subpar

## DB 3 (Justin)

### Research Question

**How has the company's client growth between 2021 and 2025 translated into revenue growth, and is that growth broad-based across different client segments or increasingly dependent on a small number of high-value clients?**

#### Graph 1

### Business Question

**How has the company's client acquisition evolved between 2021 and 2025?**

In [124]:
# ============================================================
# Cell: Graph 1 - Data Preparation
# ============================================================

# Create a copy of the client profiles dataset
client_growth = profiles.copy()

# Convert commencement date to datetime format
client_growth["COMMENCEMENT DATE"] = pd.to_datetime(
    client_growth["COMMENCEMENT DATE"]
)

# Extract commencement year
client_growth["YEAR"] = (
    client_growth["COMMENCEMENT DATE"]
    .dt.year
)

# Calculate the number of new clients acquired each year
annual_clients = (
    client_growth
    .groupby("YEAR")
    .size()
    .reindex(range(2021, 2026), fill_value=0)
    .reset_index(name="NEW CLIENTS")
)

# Calculate cumulative client growth
annual_clients["CUMULATIVE CLIENTS"] = (
    annual_clients["NEW CLIENTS"]
    .cumsum()
)

annual_clients

,YEAR,NEW CLIENTS,CUMULATIVE CLIENTS
0,2021,6,6
1,2022,15,21
2,2023,33,54
3,2024,44,98
4,2025,26,124


In [125]:
# ============================================================
# Cell: Graph 1 - Plotly Express Chart
# ============================================================

import plotly.express as px
import plotly.graph_objects as go

fig = px.bar(
    annual_clients,
    x="YEAR",
    y="NEW CLIENTS",
    text="NEW CLIENTS",
    color_discrete_sequence=["#4C78A8"],
    labels={
        "YEAR": "Year",
        "NEW CLIENTS": "Number of New Clients"
    },
    title="Client Acquisition Trend (2021–2025)"
)

fig.add_trace(
    go.Scatter(
        x=annual_clients["YEAR"],
        y=annual_clients["CUMULATIVE CLIENTS"],
        mode="lines+markers",
        name="Cumulative Clients",
        yaxis="y2",
        line=dict(
            color="#F58518",
            width=3
        ),
        marker=dict(size=8),
        hovertemplate=
        "<b>Year:</b> %{x}<br>" +
        "<b>Cumulative Clients:</b> %{y}<extra></extra>"
    )
)

fig.update_traces(
    selector=dict(type="bar"),
    textposition="outside",
    textfont=dict(size=11),
    hovertemplate=
    "<b>Year:</b> %{x}<br>" +
    "<b>New Clients:</b> %{y}<extra></extra>"
)

fig.update_layout(

    template="plotly_white",

    width=950,
    height=550,

    title=dict(
        x=0.5
    ),

    font=dict(
        family="Arial",
        size=12
    ),

    hovermode="x unified",

    xaxis=dict(
        title="Year",
        tickmode="linear"
    ),

    yaxis=dict(
        title="New Clients",
        rangemode="tozero"
    ),

    yaxis2=dict(
        title="Cumulative Clients",
        overlaying="y",
        side="right",
        showgrid=False,
        rangemode="tozero"
    ),

    legend=dict(
        orientation="h",
        y=1.05,
        x=1,
        xanchor="right"
    ),

    margin=dict(
        l=60,
        r=60,
        t=70,
        b=50
    )
)

fig.show()

### Key Insights

1. Client acquisition increased steadily between 2021 and 2024, rising from 6 new clients in 2021 to 44 new clients in 2024. This indicates that the company experienced strong expansion during the first four years.

2. Although new client acquisition declined to 26 clients in 2025, the company continued to acquire considerably more clients than in the earlier years, suggesting that growth remained positive despite slowing compared to 2024.

3. The cumulative client base increased consistently throughout the five-year period, reaching 124 clients by the end of 2025. This suggests that the company maintained continuous growth rather than experiencing short-term fluctuations.

4. While the company successfully expanded its client base, client growth alone does not necessarily indicate stronger business performance. Further analysis is required to determine whether this increase in clients translated into revenue growth.

5. The next graph examines revenue growth across different client sectors to identify which segments contributed most to the company's expansion.

#### Graph 2

### Business Question

Which client sectors contributed most to revenue growth between 2021 and 2025?

In [126]:
# ============================================================
# Cell: Graph 2 - Data Preparation
# ============================================================

# Merge revenue and client profile datasets
revenue_profiles = RC.merge(
    profiles,
    on="CLIENT ID",
    how="left"
)

# Calculate yearly revenue by sector
sector_revenue = (
    revenue_profiles
    .groupby(["YEAR", "SECTOR"], as_index=False)["REVENUE"]
    .sum()
)

# Calculate total revenue by sector across all years
sector_order = (
    sector_revenue
    .groupby("SECTOR", as_index=False)["REVENUE"]
    .sum()
    .sort_values(
        "REVENUE",
        ascending=True
    )
)

sector_order_list = sector_order["SECTOR"].tolist()

# Create all possible combinations of years and sectors
year_sector_grid = pd.MultiIndex.from_product(
    [
        sorted(sector_revenue["YEAR"].unique()),
        sector_order_list
    ],
    names=["YEAR", "SECTOR"]
).to_frame(index=False)

# Add zero revenue for sectors with no recorded revenue in a year
sector_revenue_complete = (
    year_sector_grid
    .merge(
        sector_revenue,
        on=["YEAR", "SECTOR"],
        how="left"
    )
    .fillna({"REVENUE": 0})
)

# Calculate each sector's share of annual revenue
annual_revenue = (
    sector_revenue_complete
    .groupby("YEAR")["REVENUE"]
    .transform("sum")
)

sector_revenue_complete["REVENUE SHARE"] = (
    sector_revenue_complete["REVENUE"]
    / annual_revenue
    * 100
)

In [127]:
# ============================================================
# Cell: Graph 2 - Plotly Graph Objects Chart
# ============================================================

import plotly.graph_objects as go

# Vibrant but controlled sector colours
sector_colours = {
    "Healthcare": "#2F6BFF",
    "Info Tech": "#00A896",
    "Manufacturing": "#FF9F1C",
    "Logistic": "#8B5CF6",
    "Security": "#EF476F",
    "Transportation": "#00B4D8",
    "Education": "#F15BB5",
    "Legal": "#6C63FF",
    "Finance": "#84CC16",
    "Charity": "#94A3B8"
}

years = sorted(sector_revenue_complete["YEAR"].unique())
first_year = years[0]

# Prepare the initial year
initial_data = (
    sector_revenue_complete[
        sector_revenue_complete["YEAR"] == first_year
    ]
    .set_index("SECTOR")
    .reindex(sector_order_list)
    .reset_index()
)

fig = go.Figure(
    data=[
        go.Bar(
            x=initial_data["REVENUE"],
            y=initial_data["SECTOR"],
            orientation="h",
            marker=dict(
                color=[
                    sector_colours.get(sector, "#CBD5E1")
                    for sector in initial_data["SECTOR"]
                ],
                line=dict(
                    color="white",
                    width=1
                )
            ),
            text=initial_data["REVENUE"],
            texttemplate="$%{text:,.0f}",
            textposition="outside",
            customdata=initial_data[["REVENUE SHARE"]],
            hovertemplate=(
                "<b>%{y}</b><br>"
                "Revenue: $%{x:,.0f}<br>"
                "Share of annual revenue: %{customdata[0]:.1f}%"
                "<extra></extra>"
            )
        )
    ]
)

# Create one animation frame for each year
fig.frames = [
    go.Frame(
        name=str(year),
        data=[
            go.Bar(
                x=(
                    sector_revenue_complete[
                        sector_revenue_complete["YEAR"] == year
                    ]
                    .set_index("SECTOR")
                    .reindex(sector_order_list)["REVENUE"]
                ),
                y=sector_order_list,
                orientation="h",
                marker=dict(
                    color=[
                        sector_colours.get(sector, "#CBD5E1")
                        for sector in sector_order_list
                    ],
                    line=dict(
                        color="white",
                        width=1
                    )
                ),
                text=(
                    sector_revenue_complete[
                        sector_revenue_complete["YEAR"] == year
                    ]
                    .set_index("SECTOR")
                    .reindex(sector_order_list)["REVENUE"]
                ),
                texttemplate="$%{text:,.0f}",
                textposition="outside",
                customdata=(
                    sector_revenue_complete[
                        sector_revenue_complete["YEAR"] == year
                    ]
                    .set_index("SECTOR")
                    .reindex(sector_order_list)[["REVENUE SHARE"]]
                ),
                hovertemplate=(
                    "<b>%{y}</b><br>"
                    "Revenue: $%{x:,.0f}<br>"
                    "Share of annual revenue: %{customdata[0]:.1f}%"
                    "<extra></extra>"
                )
            )
        ],
        layout=go.Layout(
            title_text=f"Revenue by Client Sector — {year}"
        )
    )
    for year in years
]

# Year slider
slider_steps = [
    dict(
        method="animate",
        label=str(year),
        args=[
            [str(year)],
            {
                "mode": "immediate",
                "frame": {
                    "duration": 500,
                    "redraw": True
                },
                "transition": {
                    "duration": 300
                }
            }
        ]
    )
    for year in years
]

fig.update_layout(
    template="plotly_white",

    width=1050,
    height=650,

    title=dict(
        text=f"Revenue by Client Sector — {first_year}",
        x=0.5
    ),

    font=dict(
        family="Arial",
        size=12
    ),

    xaxis=dict(
        title="Revenue ($)",
        tickprefix="$",
        tickformat="~s",
        rangemode="tozero",
        gridcolor="rgba(0, 0, 0, 0.08)"
    ),

    yaxis=dict(
        title="Client Sector",
        categoryorder="array",
        categoryarray=sector_order_list
    ),

    margin=dict(
        l=135,
        r=80,
        t=105,
        b=120
    ),

    sliders=[
        dict(
            active=0,
            currentvalue=dict(
                prefix="Selected year: ",
                font=dict(size=13)
            ),
            pad=dict(
                t=40
            ),
            steps=slider_steps
        )
    ],

    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            showactive=False,

            x=0,
            xanchor="left",
            y=1.15,
            yanchor="top",

            bgcolor="#E8F0FF",
            bordercolor="#2F6BFF",
            borderwidth=1,

            font=dict(
                color="#1E3A8A",
                size=12
            ),

            buttons=[
                dict(
                    label="▶ Play years",
                    method="animate",
                    args=[
                        None,
                        {
                            "fromcurrent": True,
                            "frame": {
                                "duration": 900,
                                "redraw": True
                            },
                            "transition": {
                                "duration": 350
                            }
                        }
                    ]
                ),
                dict(
                    label="Ⅱ Pause",
                    method="animate",
                    args=[
                        [None],
                        {
                            "mode": "immediate",
                            "frame": {
                                "duration": 0,
                                "redraw": False
                            },
                            "transition": {
                                "duration": 0
                            }
                        }
                    ]
                )
            ]
        )
    ]
)

fig.show()

### Key Insights

1. Revenue increased substantially between 2021 and 2025 across most client sectors, indicating that the company's revenue growth was supported by continued expansion rather than a single year of exceptional performance.

2. Healthcare generated the highest total revenue over the five-year period, followed by Information Technology and Manufacturing. These three sectors consistently contributed the largest share of the company's overall revenue.

3. Education experienced the most noticeable growth after 2023, while Legal also showed significant increases during 2024 and 2025. This suggests that the company's revenue became more diversified as additional sectors expanded.

4. Finance and Charity contributed only a small proportion of the company's total revenue throughout the five-year period, indicating that these sectors had relatively limited influence on overall revenue growth.

5. Although revenue was generated across multiple industry sectors, the stacked bar chart does not reveal whether revenue is evenly distributed among individual clients. The next graph therefore investigates whether the company's revenue is concentrated among a small number of high-value clients.

#### Graph 3

### Business Question

Is the company's revenue concentrated among a small number of high-value clients?

In [128]:
# ============================================================
# Cell: Graph 3 - Data Preparation
# ============================================================

# Store Pareto data for all years combined and for each individual year
pareto_views = {}

pareto_periods = ["All Years"] + sorted(RC["YEAR"].unique().tolist())

for period in pareto_periods:

    if period == "All Years":
        period_data = RC.copy()
    else:
        period_data = RC[
            RC["YEAR"] == period
        ].copy()

    # Calculate revenue for each client
    period_client_revenue = (
        period_data
        .groupby("CLIENT ID", as_index=False)["REVENUE"]
        .sum()
        .sort_values(
            "REVENUE",
            ascending=False
        )
        .reset_index(drop=True)
    )

    # Calculate cumulative revenue percentage
    period_total_revenue = (
        period_client_revenue["REVENUE"].sum()
    )

    period_client_revenue["CUMULATIVE REVENUE"] = (
        period_client_revenue["REVENUE"].cumsum()
    )

    period_client_revenue["CUMULATIVE %"] = (
        period_client_revenue["CUMULATIVE REVENUE"]
        / period_total_revenue
        * 100
    )

    # Calculate the number of clients required to reach 80% of revenue
    period_clients_to_80 = (
        period_client_revenue["CUMULATIVE %"]
        .ge(80)
        .idxmax()
        + 1
    )

    pareto_views[period] = {
        "data": period_client_revenue,
        "clients_to_80": period_clients_to_80,
        "client_count": len(period_client_revenue)
    }

# Retain the original all-years dataframe name for the written insights
client_revenue = pareto_views["All Years"]["data"]
clients_to_80 = pareto_views["All Years"]["clients_to_80"]

In [129]:
# ============================================================
# Cell: Graph 3 - Plotly Graph Objects Chart
# ============================================================

import plotly.graph_objects as go

# Pareto chart colours
PARETO_BLUE = "#2F6BFF"
PARETO_GREY = "#D7DEE8"
CUMULATIVE_AMBER = "#FF9F1C"
REFERENCE_RED = "#E63946"

periods = list(pareto_views.keys())
traces_per_period = 2

fig = go.Figure()

# Create bar and line traces for every selectable period
for period_index, period in enumerate(periods):

    period_data = pareto_views[period]["data"]
    period_clients_to_80 = pareto_views[period]["clients_to_80"]

    client_ranks = list(
        range(
            1,
            len(period_data) + 1
        )
    )

    period_bar_colours = [
        PARETO_BLUE
        if rank <= period_clients_to_80
        else PARETO_GREY
        for rank in client_ranks
    ]

    show_period = period_index == 0

    # Revenue bars
    fig.add_trace(
        go.Bar(
            x=client_ranks,
            y=period_data["REVENUE"],
            marker=dict(
                color=period_bar_colours,
                line=dict(
                    color="white",
                    width=0.3
                )
            ),
            name="Client Revenue",
            visible=show_period,
            customdata=period_data["CLIENT ID"],
            hovertemplate=(
                "<b>Client rank:</b> %{x}<br>"
                "<b>Client ID:</b> %{customdata}<br>"
                "<b>Revenue:</b> $%{y:,.0f}"
                "<extra></extra>"
            )
        )
    )

    # Cumulative percentage line
    fig.add_trace(
        go.Scatter(
            x=client_ranks,
            y=period_data["CUMULATIVE %"],
            mode="lines",
            line=dict(
                color=CUMULATIVE_AMBER,
                width=4
            ),
            name="Cumulative Revenue",
            yaxis="y2",
            visible=show_period,
            hovertemplate=(
                "<b>Client rank:</b> %{x}<br>"
                "<b>Cumulative revenue:</b> %{y:.1f}%"
                "<extra></extra>"
            )
        )
    )

# Build clear year-selection buttons
period_buttons = []

for period_index, period in enumerate(periods):

    period_clients_to_80 = (
        pareto_views[period]["clients_to_80"]
    )

    period_client_count = (
        pareto_views[period]["client_count"]
    )

    visibility = [
        trace_period_index == period_index
        for trace_period_index in range(len(periods))
        for _ in range(traces_per_period)
    ]

    period_buttons.append(
        dict(
            label=str(period),

            method="update",

            args=[
                {
                    "visible": visibility
                },
                {
                    "title.text": (
                        "Pareto Analysis of Client Revenue Concentration"
                        f" — {period}"
                    ),

                    "xaxis.range": [
                        0,
                        period_client_count + 1
                    ],

                    "shapes": [
                        dict(
                            type="line",
                            xref="paper",
                            x0=0,
                            x1=1,
                            yref="y2",
                            y0=80,
                            y1=80,
                            line=dict(
                                color=REFERENCE_RED,
                                dash="dash",
                                width=2
                            )
                        ),
                        dict(
                            type="line",
                            xref="x",
                            x0=period_clients_to_80,
                            x1=period_clients_to_80,
                            yref="paper",
                            y0=0,
                            y1=1,
                            line=dict(
                                color=REFERENCE_RED,
                                dash="dash",
                                width=2
                            )
                        )
                    ],

                    "annotations": [
                        dict(
                            x=period_clients_to_80,
                            y=84,
                            yref="y2",

                            text=(
                                "80% of revenue reached after"
                                f"<br>{period_clients_to_80} clients"
                            ),

                            showarrow=True,
                            arrowhead=2,
                            arrowcolor=REFERENCE_RED,

                            ax=70,
                            ay=-45,

                            bgcolor="#FFF4E6",
                            bordercolor=CUMULATIVE_AMBER,
                            borderwidth=1,

                            font=dict(
                                color="#7C2D12",
                                size=11
                            )
                        )
                    ]
                }
            ]
        )
    )

# Initial all-years values
initial_period = periods[0]
initial_clients_to_80 = (
    pareto_views[initial_period]["clients_to_80"]
)
initial_client_count = (
    pareto_views[initial_period]["client_count"]
)

fig.update_layout(
    template="plotly_white",

    width=1150,
    height=680,

    title=dict(
        text=(
            "Pareto Analysis of Client Revenue Concentration"
            f" — {initial_period}"
        ),
        x=0.5
    ),

    font=dict(
        family="Arial",
        size=12
    ),

    hovermode="x unified",

    xaxis=dict(
        title="Clients Ranked by Revenue",
        range=[
            0,
            initial_client_count + 1
        ],
        showgrid=False
    ),

    yaxis=dict(
        title="Revenue ($)",
        tickprefix="$",
        tickformat="~s",
        gridcolor="rgba(0, 0, 0, 0.08)"
    ),

    yaxis2=dict(
        title="Cumulative Revenue (%)",
        overlaying="y",
        side="right",
        range=[0, 100],
        ticksuffix="%",
        showgrid=False
    ),

    legend=dict(
        orientation="h",
        y=1.06,
        x=1,
        xanchor="right"
    ),

    margin=dict(
        l=75,
        r=85,
        t=120,
        b=65
    ),

    # Distinct button row
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            buttons=period_buttons,

            x=0.5,
            xanchor="center",
            y=1.17,
            yanchor="top",

            showactive=True,
            active=0,

            bgcolor="#E8F0FF",
            bordercolor=PARETO_BLUE,
            borderwidth=1,

            font=dict(
                color="#1E3A8A",
                size=11
            ),

            pad=dict(
                r=5,
                t=5
            )
        )
    ],

    shapes=[
        dict(
            type="line",
            xref="paper",
            x0=0,
            x1=1,
            yref="y2",
            y0=80,
            y1=80,
            line=dict(
                color=REFERENCE_RED,
                dash="dash",
                width=2
            )
        ),
        dict(
            type="line",
            xref="x",
            x0=initial_clients_to_80,
            x1=initial_clients_to_80,
            yref="paper",
            y0=0,
            y1=1,
            line=dict(
                color=REFERENCE_RED,
                dash="dash",
                width=2
            )
        )
    ],

    annotations=[
        dict(
            x=initial_clients_to_80,
            y=84,
            yref="y2",

            text=(
                "80% of revenue reached after"
                f"<br>{initial_clients_to_80} clients"
            ),

            showarrow=True,
            arrowhead=2,
            arrowcolor=REFERENCE_RED,

            ax=70,
            ay=-45,

            bgcolor="#FFF4E6",
            bordercolor=CUMULATIVE_AMBER,
            borderwidth=1,

            font=dict(
                color="#7C2D12",
                size=11
            )
        )
    ]
)

fig.show()

### Key Insights

1. The Pareto analysis shows that approximately 66 out of 140 clients (around 47% of the client base) contributed to 80% of the company's total revenue between 2021 and 2025.

2. This indicates that the company's revenue is not concentrated among only a small number of high-value clients. Instead, revenue is generated across a relatively broad group of clients, reducing dependence on any single customer or small client group.

3. Although several clients generated substantially higher revenue than others, the cumulative revenue increased gradually rather than sharply. This suggests that many medium-sized clients also made meaningful contributions to the company's overall revenue.

4. A more diversified revenue base generally reduces business risk, as the loss of an individual client is less likely to have a significant impact on the company's overall revenue performance.

5. Based on the three analyses, the company experienced steady client acquisition, revenue growth across multiple industry sectors, and a relatively diversified revenue base. These findings suggest that the company's growth between 2021 and 2025 was supported by broad client expansion rather than relying solely on a small number of major clients.